# Gold local - modelagem dimensional com Delta

In [ ]:
import os
from pathlib import Path
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

java_home = Path("/usr/lib/jvm/java-17-openjdk-amd64")
if java_home.exists():
    os.environ["JAVA_HOME"] = str(java_home)
    os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")
else:
    raise RuntimeError("JDK 17 não encontrado em /usr/lib/jvm/java-17-openjdk-amd64. Instale o OpenJDK 17 para rodar o Spark localmente.")

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / "landing" / "dados").exists():
        project_root = candidate
        break

workspace_dir = (project_root / "spark-warehouse").resolve()
caminho_silver = (project_root / "silver").resolve()
caminho_gold = (project_root / "gold").resolve()

os.makedirs(caminho_gold, exist_ok=True)
os.makedirs(workspace_dir, exist_ok=True)

try:
    current = SparkSession.getActiveSession()
    if current is not None:
        current.stop()
except Exception:
    pass

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Projeto_Medalhao_Local_Gold")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", str(workspace_dir))
)

print(f"Gold: {caminho_gold}")
print(f"Silver: {caminho_silver}")
print(f"Project root: {project_root}")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

IndentationError: unexpected indent (176032657.py, line 31)

In [ ]:
# Leitura das tabelas Silver em Delta local
df_students = spark.read.format("delta").load(str(caminho_silver / "students"))
df_teacher = spark.read.format("delta").load(str(caminho_silver / "teacher"))
df_classes = spark.read.format("delta").load(str(caminho_silver / "classes"))
df_subjects = spark.read.format("delta").load(str(caminho_silver / "subjects"))
df_grades = spark.read.format("delta").load(str(caminho_silver / "grades"))

print("Tabelas Silver lidas com sucesso")


In [ ]:
from pyspark.sql import functions as F


def drop_audit_columns(df):
    cols_to_remove = [c for c in df.columns if c.upper() in {"DATA_HORA_SILVER", "NOME_TABELA"}]
    return df.drop(*cols_to_remove) if cols_to_remove else df


def first_matching_column(df, candidates):
    upper_map = {c.upper(): c for c in df.columns}
    for candidate in candidates:
        if candidate.upper() in upper_map:
            return upper_map[candidate.upper()]
    return None


def build_dimension(df, id_candidates, name_candidates, id_alias, name_alias):
    id_col = first_matching_column(df, id_candidates)
    name_col = first_matching_column(df, name_candidates)

    if id_col is None:
        raise ValueError(f"Nenhuma coluna de identificador encontrada. Candidatos: {id_candidates}")

    selected = [F.col(id_col).alias(id_alias)]
    if name_col is not None:
        selected.append(F.col(name_col).alias(name_alias))
    else:
        selected.append(F.lit(None).cast("string").alias(name_alias))

    return df.select(*selected)


df_students = drop_audit_columns(df_students)
df_teacher = drop_audit_columns(df_teacher)
df_classes = drop_audit_columns(df_classes)
df_subjects = drop_audit_columns(df_subjects)
df_grades = drop_audit_columns(df_grades)

print("Colunas de auditoria removidas")


In [ ]:
# Construção das dimensões
# O schema da camada Silver usa nomes como IDENTIFIER, STUDENT_IDENTIFIER, TEACHER_IDENTIFIER...
dim_student = build_dimension(
    df_students,
    ["IDENTIFIER", "STUDENT_IDENTIFIER", "ID_STUDENT", "STUDENT_ID", "IDENTIFIER_STUDENT"],
    ["NAME", "NOME", "NOME_COMPLETO", "NOME_ESTUDANTE"],
    "student_id",
    "student_name"
)

dim_teacher = build_dimension(
    df_teacher,
    ["IDENTIFIER", "TEACHER_IDENTIFIER", "ID_TEACHER", "TEACHER_ID", "IDENTIFIER_TEACHER"],
    ["NAME", "NOME", "NOME_PROFESSOR", "NOME_COMPLETO"],
    "teacher_id",
    "teacher_name"
)

dim_class = build_dimension(
    df_classes,
    ["IDENTIFIER", "CLASS_IDENTIFIER", "ID_CLASS", "CLASS_ID", "IDENTIFIER_CLASS"],
    ["NAME", "NOME", "NOME_TURMA", "DESCRICAO"],
    "class_id",
    "class_name"
)

dim_subject = build_dimension(
    df_subjects,
    ["IDENTIFIER", "SUBJECT_IDENTIFIER", "ID_SUBJECT", "SUBJECT_ID", "IDENTIFIER_SUBJECT"],
    ["NAME", "NOME", "NOME_DISCIPLINA", "DESCRICAO"],
    "subject_id",
    "subject_name"
)

print("Dimensões montadas")


In [ ]:
# Construção da tabela de fato
# A camada Silver converteu os IDs para nomes do tipo STUDENT_IDENTIFIER, TEACHER_IDENTIFIER...
student_fk = first_matching_column(df_grades, ["STUDENT_IDENTIFIER", "IDENTIFIER", "ID_STUDENT", "STUDENT_ID", "IDENTIFIER_STUDENT"])
teacher_fk = first_matching_column(df_grades, ["TEACHER_IDENTIFIER", "IDENTIFIER", "ID_TEACHER", "TEACHER_ID", "IDENTIFIER_TEACHER"])
class_fk = first_matching_column(df_grades, ["CLASS_IDENTIFIER", "IDENTIFIER", "ID_CLASS", "CLASS_ID", "IDENTIFIER_CLASS"])
subject_fk = first_matching_column(df_grades, ["SUBJECT_IDENTIFIER", "IDENTIFIER", "ID_SUBJECT", "SUBJECT_ID", "IDENTIFIER_SUBJECT"])
grade_value = first_matching_column(df_grades, ["GRADE", "NOTA", "GRADE_VALUE", "NOTA_FINAL", "VALOR_NOTA"])

if None in {student_fk, teacher_fk, class_fk, subject_fk, grade_value}:
    raise ValueError("Uma ou mais colunas essenciais da tabela grades não foram encontradas.")

fact_grades = df_grades.select(
    F.col(student_fk).alias("student_id"),
    F.col(teacher_fk).alias("teacher_id"),
    F.col(class_fk).alias("class_id"),
    F.col(subject_fk).alias("subject_id"),
    F.col(grade_value).cast("double").alias("grade_value")
)

print("Tabela fato montada")


In [ ]:
# Escrita em Gold no formato Delta local
# As pastas de saída não usam Unity Catalog, apenas arquivos no filesystem local.
dim_student.write.format("delta").mode("overwrite").save(str(caminho_gold / "dim_student"))
dim_teacher.write.format("delta").mode("overwrite").save(str(caminho_gold / "dim_teacher"))
dim_class.write.format("delta").mode("overwrite").save(str(caminho_gold / "dim_class"))
dim_subject.write.format("delta").mode("overwrite").save(str(caminho_gold / "dim_subject"))
fact_grades.write.format("delta").mode("overwrite").save(str(caminho_gold / "fact_grades"))

print("Tabelas do Gold gravadas com sucesso")


In [ ]:
import os
print("Tabelas na pasta Gold local:")
print(sorted(os.listdir(caminho_gold)))
